In [1]:
import torch
import torch.nn as nn
torch.manual_seed(42)

## Model

In [2]:
class SimpleDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.convtrans = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)
        self.convblock = nn.Conv2d(2, 1, kernel_size=1)

    def forward(self, x_bottom, x_skip):
        x_up = self.convtrans(x_bottom)
        x_concat = torch.cat([x_skip, x_up], dim=1)
        x_out = self.convblock(x_concat)
        return x_out


class TwoStageDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = SimpleDecoder()   # 1x1 -> 2x2
        self.stage2 = SimpleDecoder()   # 2x2 -> 4x4

    def forward(self, x_bottleneck, x_skip1, x_skip2):
        x_out1 = self.stage1(x_bottleneck, x_skip1)
        x_out2 = self.stage2(x_out1, x_skip2)
        return x_out1, x_out2

In [3]:
model = TwoStageDecoder()

## Set fixed weights

In [4]:
# Gợi ý:
# theo công thức ta có: weight shape = [out_channel, in_channel, row, col]
#
# Bài này chỉ có:
# - 1 output channel
# - 2 input channels  -> in_channel tại kênh 0 là vị trí có weights để tính convolution cho giá trị từ skip-connection, tương tự in_channel tại kênh 1 là dành cho giá trị sau Convolution transposed
# - kernel 1x1        -> row = 1, col = 1
# --> convblock.weight.shape = [1, 2, 1, 1]

# Có thể hình dung weight của convolution block nếu set in_channel tại kênh 0 = 1.0 và in_channel tại kênh 1 = 2.0 như sau:
# weight =
# [
#   [
#     [[1.0]],
#     [[2.0]]
#   ]
# ]
#

# Khởi tạo ma trận skip connection
x_skip1 = torch.randint(low=0, high=5, size=(1, 1, 2, 2)).float()
x_skip2 = torch.randint(low=0, high=5, size=(1, 1, 4, 4)).float()

# -----------------------------
# Set weight cố định
# -----------------------------
with torch.no_grad():
    # ===== Stage 1 =====
    # ConvTranspose2d: weight shape = [1, 1, 2, 2]
    nn.init.xavier_uniform_(model.stage1.convtrans.weight)

    # Conv2d: weight shape = [1, 2, 1, 1]
    model.stage1.convblock.weight.zero_()
    # input channel 0 = x_skip1
    # input channel 1 = x_up từ x_bottleneck
    model.stage1.convblock.weight[0, 0, 0, 0] = 1.0
    model.stage1.convblock.weight[0, 1, 0, 0] = 2.0

    # ===== Stage 2 =====
    # ConvTranspose2d: weight shape = [1, 1, 2, 2]
    nn.init.xavier_uniform_(model.stage2.convtrans.weight)

    # Conv2d: weight shape = [1, 2, 1, 1]
    # input channel 0 = x_skip2
    # input channel 1 = x_up từ x_out1
    model.stage2.convblock.weight.zero_()
    model.stage2.convblock.weight[0, 0, 0, 0] = 1.0
    model.stage2.convblock.weight[0, 1, 0, 0] = 2.0

In [7]:
x_bottleneck = torch.tensor([[2.]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
x_bottleneck

tensor([[[[2.]]]])

## Forward

In [6]:
# -----------------------------
# Forward
# -----------------------------
x_out1, x_out2 = model(x_bottleneck, x_skip1, x_skip2)

print("x_bottleneck:")
print(x_bottleneck.squeeze())

print("\nx_skip1:")
print(x_skip1.squeeze())

print("\nx_skip2:")
print(x_skip2.squeeze())

print("\nx_out1:")
print(x_out1.squeeze())

print("\nx_out2:")
print(x_out2.squeeze())

x_bottleneck:
tensor(2.)

x_skip1:
tensor([[0., 0.],
        [2., 1.]])

x_skip2:
tensor([[4., 1., 3., 1.],
        [4., 3., 1., 4.],
        [2., 4., 2., 0.],
        [0., 4., 3., 4.]])

x_out1:
tensor([[0.7361, 2.9950],
        [2.5749, 0.1037]], grad_fn=<SqueezeBackward0>)

x_out2:
tensor([[4.9815, 1.2933, 5.0328, 0.2322],
        [5.1755, 4.7772, 3.8222, 9.2705],
        [3.8372, 3.4296, 2.6872, 0.5903],
        [2.5159, 8.6208, 3.7146, 4.7993]], grad_fn=<SqueezeBackward0>)
